# CUDA GPU Deep Learning Benchmark

## tl;dr

- GPU tested: NVIDIA GeForce RTX 5050 Laptop GPU with 7.96 GB VRAM.
- PyTorch saw CUDA successfully: True (torch 2.11.0+cu128, CUDA wheel 12.8).
- Best synthetic ConvNet training result: 400.8 images/sec (159.7 ms/step), batch 64, amp_float16.
- Best GPU training throughput was 35.5x the small CPU baseline (11.3 images/sec).
- Best matrix multiply result: 6.45 TFLOP/s (float16, size 1024).

## Context & Methods

This notebook uses synthetic image batches so the timing focuses on GPU compute rather than network downloads, disk reads, or data preprocessing. The training benchmark runs a compact convolutional network in both float32/TF32-enabled mode and CUDA AMP float16 mode, then compares the best GPU result with a small CPU baseline.

### Key Assumptions

- Synthetic data measures training compute throughput, not end-to-end data-loader performance.
- Laptop GPU performance depends strongly on power mode, thermals, and whether the machine is plugged in.
- The CUDA wheel is cu128; the installed NVIDIA driver advertises CUDA 12.9 and is backward-compatible with this wheel.

## Setup

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

from IPython.display import Markdown, display

ROOT = Path.cwd()
SCRIPTS_DIR = ROOT / "scripts"
RESULTS_DIR = ROOT / "results"
RESULTS_JSON = RESULTS_DIR / "benchmark_results.json"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import gpu_benchmark

CONFIG = gpu_benchmark.merge_config()
CONFIG

## Data

In [ ]:
environment = gpu_benchmark.collect_environment()

compact_environment = {
    key: value
    for key, value in environment.items()
    if key != "nvidia_smi"
}

print(json.dumps(compact_environment, indent=2))
print("\n--- nvidia-smi ---")
print(environment["nvidia_smi"])

## Results

In [ ]:
payload = gpu_benchmark.run_all_benchmarks(config=CONFIG)
gpu_benchmark.print_summary(payload)

In [ ]:
def markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        values = []
        for column in columns:
            value = row.get(column, "")
            if isinstance(value, float):
                value = f"{value:.3f}"
            values.append(str(value))
        body.append("| " + " | ".join(values) + " |")
    return "\n".join([header, divider, *body])


training_columns = [
    "device",
    "precision",
    "batch_size",
    "timed_steps",
    "ms_per_step",
    "images_per_second",
    "peak_memory_gb",
    "error",
]

display(Markdown("### Training Throughput"))
display(Markdown(markdown_table(payload["training_results"], training_columns)))

In [ ]:
matrix_columns = [
    "dtype",
    "matrix_size",
    "repeats",
    "elapsed_seconds",
    "tflops",
    "error",
]

display(Markdown("### Matrix Multiply Throughput"))
display(Markdown(markdown_table(payload["matrix_results"], matrix_columns)))

In [ ]:
import matplotlib.pyplot as plt

plot_rows = [
    row
    for row in payload["training_results"]
    if row.get("device") == "cuda" and "images_per_second" in row
]

if plot_rows:
    labels = [f"{row['precision']}\nbatch {row['batch_size']}" for row in plot_rows]
    values = [row["images_per_second"] for row in plot_rows]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(labels, values, color=["#3772ff" if "amp" in label else "#2d936c" for label in labels])
    ax.set_title("Synthetic ConvNet Training Throughput")
    ax.set_ylabel("Images per second")
    ax.tick_params(axis="x", labelrotation=30)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    chart_path = RESULTS_DIR / "training_throughput.png"
    fig.savefig(chart_path, dpi=150)
    plt.show()
    print(f"Saved chart to {chart_path}")
else:
    print("No CUDA training rows were available to plot.")

## Takeaways

In [ ]:
summary = payload["summary"]
best_training = summary.get("best_cuda_training")
best_cpu = summary.get("best_cpu_training")
best_matrix = summary.get("best_matrix_multiply")
speedup = summary.get("training_speedup_vs_cpu")

takeaways = []
if best_training:
    takeaways.append(
        f"- Best GPU training result was **{best_training['images_per_second']:.1f} images/sec** "
        f"at batch {best_training['batch_size']} using `{best_training['precision']}`."
    )
if best_cpu and speedup:
    takeaways.append(
        f"- That was **{speedup:.1f}x faster** than the CPU baseline in this notebook "
        f"({best_cpu['images_per_second']:.1f} images/sec)."
    )
if best_matrix:
    takeaways.append(
        f"- Peak matrix multiply throughput was **{best_matrix['tflops']:.2f} TFLOP/s** "
        f"for `{best_matrix['dtype']}` at size {best_matrix['matrix_size']}."
    )
if not takeaways:
    takeaways.append("- CUDA results were not produced; check the environment output above.")

display(Markdown("\n".join(takeaways)))